# Data Loading, Cleaning & Feature Engineering

Bike Share Toronto 2023 Ridership Analysis

In [12]:
import sys
from pathlib import Path
import pandas as pd

# Ensure that the project root is on the path so src/ can be imported
PROJECT_ROOT = Path().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.load_data_multi import load_bike_ridership

df = load_bike_ridership(years=list(range(2020, 2026)))

Total rows loaded: 30,521,083
Successfully loaded 30,521,083 rows.


In [3]:
df.head()

,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type,year
Trip Id,,,,,,,,,,
7334128,648,7003,01/01/2020 00:08,Madison Ave / Bloor St W,7271.0,01/01/2020 00:19,Yonge St / Alexander St - SMART,3104,Annual Member,2020
7334129,419,7007,01/01/2020 00:10,College St / Huron St,7163.0,01/01/2020 00:17,Yonge St / Wood St,2126,Annual Member,2020
7334130,566,7113,01/01/2020 00:13,Parliament St / Aberdeen Ave,7108.0,01/01/2020 00:22,Front St E / Cherry St,4425,Annual Member,2020
7334131,1274,7333,01/01/2020 00:17,King St E / Victoria St,7311.0,01/01/2020 00:38,Sherbourne St / Isabella St,4233,Annual Member,2020
7334132,906,7009,01/01/2020 00:19,King St E / Jarvis St,7004.0,01/01/2020 00:34,University Ave / Elm St,2341,Casual Member,2020


In [ ]:
df.tail()

,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type,year
Trip Id,,,,,,,,,,
43436831,377,7271,2025-11-30 23:55:58,Yonge St / Alexander St - SMART,7270.0,2025-12-01 00:02:15,Church St / Dundas St E - SMART,9264,Casual Member,2025
43436832,164,7281,2025-11-30 23:56:00,Charles St W / Balmuto St - SMART,8118.0,2025-11-30 23:58:44,Clover Hill Park - SMART,8834,Annual Member,2025
43436833,776,7543,2025-11-30 23:56:24,Nassau St / Bellevue Ave,7259.0,2025-12-01 00:09:20,Lower Spadina Ave / Lake Shore Blvd W,7240,Annual Member,2025
43436836,1471,7719,2025-11-30 23:59:29,Wolseley St / Augusta Ave,7181.0,2025-12-01 00:24:00,Lansdowne Ave / Whytock Ave,9370,Annual Member,2025
43436838,223,7272,2025-11-30 23:59:46,Yonge St / Dundonald St,7271.0,2025-12-01 00:03:29,Yonge St / Alexander St - SMART,8720,Annual Member,2025


### Two-Stage Filtering Strategy

The SQL query above is an intentional first-pass filter applied at load time. It removes:

- Trips under 60 seconds are almost always docking errors (bike undocked and immediately re-docked).
- Trips over 3600 seconds (1 hour) are statistical outliers that skew the distribution and are unlikely to represent typical commuter or recreational rides.

This reduces the dataset from 5.7M to around 5.6M rows before any statistical analysis.

A second-pass filter using the IQR method (below) then removes remaining statistical outliers from the already-truncated distribution. The IQR bounds reported below are therefore computed on the pre-filtered data, not the raw dataset.

In [ ]:
# Column types and missing values
print("Columns:", df.columns.tolist())
print("\nMissing values:")
print(df.isna().sum())
print(f"\nUnique start stations: {df['Start Station Name'].nunique()}")
print(f"Unique end stations: {df['End Station Name'].nunique()}")

Columns: ['Trip Duration', 'Start Station Id', 'Start Time', 'Start Station Name', 'End Station Id', 'End Time', 'End Station Name', 'Bike Id', 'User Type', 'year']

Missing values:
Trip Duration              0
Start Station Id           1
Start Time                 0
Start Station Name    743382
End Station Id          8404
End Time                   0
End Station Name      750794
Bike Id                  275
User Type                  0
year                       0
dtype: int64

Unique start stations: 1399
Unique end stations: 1400


Small note on missing station names: Around 10% of trips are missing Start/End Station Name. These rows still have valid Station IDs, durations, and timestamps, so we retain them for temporal analysis but they cannot be used for any station-level work.

### Feature Engineering
Parse timestamps, extract temporal components (hour, day, month), create Weekday/Weekend and Peak Hour flags, and convert Trip Duration to minutes.

Peak hours are defined as Morning (7:00 to 9:59) and Evening (15:00 to 19:59), while everything else in off peak.

In [ ]:
from src.feature_engineering import parse_timestamps, add_derived_features

df_clean = parse_timestamps(df)
df_clean = add_derived_features(df_clean)

print("Columns after feature engineering:")
print(df_clean.columns.tolist())

del df

Columns after feature engineering:
['Trip Duration', 'Start Station Id', 'Start Time', 'Start Station Name', 'End Station Id', 'End Time', 'End Station Name', 'Bike Id', 'User Type', 'year', 'Start Date', 'Start Hour', 'End Date', 'Trip_Duration_Min', 'Peak_Hour_Binary', 'User_Type_Binary']


In [7]:
df_clean.head()

,Trip Duration,Start Station Id,Start Time,Start Station Name,End Station Id,End Time,End Station Name,Bike Id,User Type,year,Start Date,Start Hour,End Date,Trip_Duration_Min,Peak_Hour_Binary,User_Type_Binary
Trip Id,,,,,,,,,,,,,,,,
7334128,648,7003,00:08,Madison Ave / Bloor St W,7271.0,00:19,Yonge St / Alexander St - SMART,3104,Annual Member,2020,2020-01-01,0,2020-01-01,10.800000,0,1
7334129,419,7007,00:10,College St / Huron St,7163.0,00:17,Yonge St / Wood St,2126,Annual Member,2020,2020-01-01,0,2020-01-01,6.983333,0,1
7334130,566,7113,00:13,Parliament St / Aberdeen Ave,7108.0,00:22,Front St E / Cherry St,4425,Annual Member,2020,2020-01-01,0,2020-01-01,9.433333,0,1
7334131,1274,7333,00:17,King St E / Victoria St,7311.0,00:38,Sherbourne St / Isabella St,4233,Annual Member,2020,2020-01-01,0,2020-01-01,21.233333,0,1
7334132,906,7009,00:19,King St E / Jarvis St,7004.0,00:34,University Ave / Elm St,2341,Casual Member,2020,2020-01-01,0,2020-01-01,15.100000,0,0


### Missing Values

In [ ]:
# Drop trips with no end location (incomplete trips)
before = len(df_clean)
df_clean = df_clean.dropna(subset=['End Station Id'])
after = len(df_clean)
print(f"Dropped {before - after:,} incomplete trips")
print(f"Remaining: {after:,} rows")

Dropped 8,404 incomplete trips (missing End Station Id)
Remaining: 30,512,679 rows


In [9]:
# Verify: only station names should have NaNs now
nan_counts = df_clean.isna().sum()
nan_counts = nan_counts[nan_counts > 0]
if len(nan_counts) == 0:
    print("No missing values remain.")
else:
    print("Remaining missing values:")
    print(nan_counts)

Remaining missing values:
Start Station Name    743177
End Station Name      745345
Bike Id                  275
dtype: int64


### Export Processed Data

In [15]:
# Export to parquet (preserving Trip Id index)
output_path = PROJECT_ROOT / 'outputs' / 'data' / 'ridership_clean.parquet'

df_clean['Start Station Id'] = pd.to_numeric(df_clean['Start Station Id'], errors='coerce')
df_clean['End Station Id'] = pd.to_numeric(df_clean['End Station Id'], errors='coerce')
df_clean['Bike Id'] = pd.to_numeric(df_clean['Bike Id'], errors='coerce')
df_clean['Trip Duration'] = pd.to_numeric(df_clean['Trip Duration'], errors='coerce')

# Use snappy compression for a balance of speed and size
# index=True keeps Trip Id in the file for traceability
df_clean.to_parquet(output_path, compression='snappy', index=True)

del df_clean